In [ ]:
# Cell 1 — Auto-install missing packages
import sys, subprocess

REQUIRED = [
    "pandas", "numpy", "scipy", "matplotlib", "seaborn",
    "plotly", "kaleido", "requests", "nbconvert", "weasyprint"
]
for pkg in REQUIRED:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ All packages available.")

In [ ]:
# Cell 2 — Core imports & palette
import warnings
warnings.filterwarnings("ignore")

import json, os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import requests
from scipy import stats
from IPython.display import display, HTML

# Project paths
NOTEBOOK_DIR = pathlib.Path(".").resolve()
ROOT         = NOTEBOOK_DIR.parent
DATA_RAW     = ROOT / "data" / "raw"
ASSETS       = ROOT / "assets"
DATA_RAW.mkdir(parents=True, exist_ok=True)
ASSETS.mkdir(parents=True, exist_ok=True)

# World Cup colour palette
WC_GREEN  = "#1a6b3c"
WC_GOLD   = "#FFD700"
WC_SAGE   = "#e8f5e0"
WC_DARK   = "#0d4a28"
WC_RED    = "#e74c3c"
WC_BLUE   = "#4da6ff"
WC_WHITE  = "#ffffff"

# Matplotlib global style
plt.rcParams.update({
    "figure.facecolor":  WC_DARK,
    "axes.facecolor":    WC_DARK,
    "text.color":        "white",
    "axes.labelcolor":   "white",
    "xtick.color":       "white",
    "ytick.color":       "white",
    "axes.edgecolor":    WC_GREEN,
    "grid.color":        "#2d8c55",
    "grid.alpha":        0.3,
    "grid.linestyle":    "--",
    "font.family":       "sans-serif",
    "font.size":         11,
})

print(f"✅ Imports complete.")
print(f"   ROOT   : {ROOT}")
print(f"   DATA   : {DATA_RAW}")
print(f"   ASSETS : {ASSETS}")

<div style="background: linear-gradient(135deg, #1a6b3c 0%, #0d4a28 50%, #1a6b3c 100%);
            border: 3px solid #FFD700; border-radius: 18px; padding: 36px 48px;
            margin: 20px 0; font-family: 'Helvetica Neue', Arial, sans-serif;
            text-align: center;
            box-shadow: 0 6px 28px rgba(255, 215, 0, 0.25);">
  <div style="font-size: 52px; margin-bottom: 12px; letter-spacing: 8px;">⚽ &nbsp; 🏆 &nbsp; ⚽</div>
  <h1 style="color: #FFD700; font-size: 2.6em; font-weight: 800;
             letter-spacing: 2px; margin: 0 0 12px 0;
             text-shadow: 2px 3px 6px rgba(0,0,0,0.6);">
    World Cup Baby Boom Analysis
  </h1>
  <p style="color: #e8f5e0; font-size: 1.25em; margin: 0 0 20px 0; opacity: 0.92; font-style: italic;">
    Does winning the FIFA World Cup trigger a national birth rate surge 9 months later?
  </p>
  <div style="display: inline-flex; gap: 16px; flex-wrap: wrap; justify-content: center;">
    <span style="padding: 7px 18px; background: rgba(255,215,0,0.12);
                border: 1px solid rgba(255,215,0,0.45); border-radius: 20px;
                color: #FFD700; font-size: 0.88em; font-weight: 600;">FIFA 1930–2022</span>
    <span style="padding: 7px 18px; background: rgba(255,215,0,0.12);
                border: 1px solid rgba(255,215,0,0.45); border-radius: 20px;
                color: #FFD700; font-size: 0.88em; font-weight: 600;">World Bank API</span>
    <span style="padding: 7px 18px; background: rgba(255,215,0,0.12);
                border: 1px solid rgba(255,215,0,0.45); border-radius: 20px;
                color: #FFD700; font-size: 0.88em; font-weight: 600;">Happiness Index</span>
  </div>
</div>

## Background

The FIFA World Cup is the most-watched sporting event on Earth, drawing over 1.5 billion viewers for major matches. When a nation wins, the collective euphoria — street celebrations, national pride, a sense of shared triumph — is hypothesised to elevate conception rates, producing a measurable birth rate increase approximately **9 months later**.

This analysis tests that hypothesis using:

- **Birth rate data** from the World Bank (crude birth rate per 1,000 people, 1960–2024)
- **World Cup results** for all 22 tournaments (1930–2022), with winners and host nations
- **Happiness scores** from Our World in Data / World Happiness Report (2011–2024)

We analyse **winners** and **host nations** separately, and examine whether the magnitude
of the birth rate effect correlates with national happiness levels.

---

> **Why Y+1?** World Cup finals are held in late June or early July. Conceptions peaking
> in July–August result in births in April–May of the following year. Annual birth rate
> data captures these births in year **Y+1**, making it the best available proxy using
> country-level annual statistics.

## Hypothesis

**Null Hypothesis ($H_0$):**

$$\Delta BR_i = BR_{i,\,Y+1} - \overline{BR}_{i,\,Y-5:Y-1} = 0$$

The crude birth rate in the year after a World Cup win does not differ from
the 5-year pre-win rolling average for winning country $i$.

**Alternative Hypothesis ($H_1$):**

$$\Delta BR_i > 0$$

Winning countries show a **positive** birth rate delta in the year following the win.

**Test:** One-sample $t$-test of all deltas against $\mu_0 = 0$, two-tailed.

## Methodology

| Step | Action |
|:----:|--------|
| 1 | Identify winning country $i$ for each World Cup year $Y$ |
| 2 | Fetch crude birth rate (World Bank, `SP.DYN.CBRT.IN`) for country $i$ |
| 3 | Compute 5-year baseline: $\overline{BR}_{i,\,Y-5:Y-1}$ (min. 3 years required) |
| 4 | Compute delta: $\Delta BR_i = BR_{i,\,Y+1} - \overline{BR}_{i,\,Y-5:Y-1}$ |
| 5 | Compute % change: $\frac{\Delta BR_i}{\overline{BR}_{i,\,Y-5:Y-1}} \times 100$ |
| 6 | Repeat for host nations |
| 7 | Merge with Happiness Index for post-2011 tournaments |
| 8 | Run correlation analysis (Pearson + Spearman) |

**Data scope:** World Bank birth rate data begins in 1960. Wins in 1930–1958 are excluded.
The 1962 Brazil win has a partial baseline (only 3 of 5 pre-win years available).

<div style="background: linear-gradient(90deg, #1a6b3c, #0d4a28);
            border-left: 6px solid #FFD700; border-radius: 10px;
            padding: 18px 28px; margin: 28px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <h2 style="color: #FFD700; margin: 0; font-size: 1.55em; font-weight: 700; letter-spacing: 1px;">
    ⚽ &nbsp; Section 2 &nbsp;|&nbsp; Data Loading &amp; Cleaning
  </h2>
</div>

In [ ]:
# Cell 8 — World Cup winners & hosts (hardcoded)
WC_RAW = [
    # (year, winner, winner_iso, host, host_iso, host_and_win)
    (1930, "Uruguay",      "URY", "Uruguay",       "URY", True),
    (1934, "Italy",        "ITA", "Italy",          "ITA", True),
    (1938, "Italy",        "ITA", "France",         "FRA", False),
    (1950, "Uruguay",      "URY", "Brazil",         "BRA", False),
    (1954, "West Germany", "DEU", "Switzerland",    "CHE", False),
    (1958, "Brazil",       "BRA", "Sweden",         "SWE", False),
    (1962, "Brazil",       "BRA", "Chile",          "CHL", False),
    (1966, "England",      "GBR", "England",        "GBR", True),
    (1970, "Brazil",       "BRA", "Mexico",         "MEX", False),
    (1974, "West Germany", "DEU", "West Germany",   "DEU", True),
    (1978, "Argentina",    "ARG", "Argentina",      "ARG", True),
    (1982, "Italy",        "ITA", "Spain",          "ESP", False),
    (1986, "Argentina",    "ARG", "Mexico",         "MEX", False),
    (1990, "West Germany", "DEU", "Italy",          "ITA", False),
    (1994, "Brazil",       "BRA", "United States",  "USA", False),
    (1998, "France",       "FRA", "France",         "FRA", True),
    (2002, "Brazil",       "BRA", "Japan",          "JPN", False),
    (2002, "Brazil",       "BRA", "South Korea",    "KOR", False),
    (2006, "Italy",        "ITA", "Germany",        "DEU", False),
    (2010, "Spain",        "ESP", "South Africa",   "ZAF", False),
    (2014, "Germany",      "DEU", "Brazil",         "BRA", False),
    (2018, "France",       "FRA", "Russia",         "RUS", False),
    (2022, "Argentina",    "ARG", "Qatar",          "QAT", False),
]

df_wc = pd.DataFrame(WC_RAW, columns=[
    "year", "winner", "winner_iso", "host", "host_iso", "host_and_win"
])

# Winners — one row per tournament (drop 2002 duplicate host row)
df_winners = df_wc.drop_duplicates(subset="year", keep="first").copy().reset_index(drop=True)

# Hosts — all rows (2002 has two)
df_hosts = df_wc[["year", "host", "host_iso", "host_and_win"]].drop_duplicates().reset_index(drop=True)

# Display name normalisation
NORM = {"West Germany": "Germany", "England": "England (UK)"}
df_winners["winner_display"] = df_winners["winner"].map(lambda x: NORM.get(x, x))

# Win count per ISO
win_counts = df_winners.groupby("winner_iso").size().rename("wc_win_count")
df_winners = df_winners.merge(win_counts, on="winner_iso", how="left")

print(f"Tournaments: {df_winners['year'].nunique()}")
print(f"Unique winning nations: {df_winners['winner_iso'].nunique()}")
print(f"Host + Win occasions: {df_winners['host_and_win'].sum()}")
display(df_winners[["year", "winner_display", "winner_iso", "host", "wc_win_count", "host_and_win"]])

In [ ]:
# Cell 9 — Styled WC results table
def style_wc_table(df, caption=""):
    """Apply World Cup green/gold styling."""
    styled = (
        df.style
          .set_table_styles([
              {"selector": "table",
               "props": [("border-collapse", "separate"),
                         ("border-spacing", "0"),
                         ("border-radius", "10px"),
                         ("overflow", "hidden"),
                         ("width", "100%"),
                         ("font-family", "'Helvetica Neue', Arial, sans-serif")]},
              {"selector": "thead th",
               "props": [("background-color", WC_GREEN),
                         ("color", WC_GOLD),
                         ("font-weight", "700"),
                         ("padding", "10px 14px"),
                         ("border-bottom", f"2px solid {WC_GOLD}"),
                         ("text-align", "center")]},
              {"selector": "tbody td",
               "props": [("padding", "8px 14px"),
                         ("border-bottom", "1px solid #c8e6c9")]},
              {"selector": "tbody tr:nth-child(even)",
               "props": [("background-color", WC_SAGE)]},
              {"selector": "tbody tr:hover",
               "props": [("background-color", "#d4edda")]},
          ])
          .hide(axis="index")
    )
    if caption:
        styled = styled.set_caption(caption)
    return styled

display(style_wc_table(
    df_winners[["year", "winner_display", "winner_iso", "host", "wc_win_count", "host_and_win"]]
    .rename(columns={
        "year": "Year", "winner_display": "Winner", "winner_iso": "ISO",
        "host": "Host", "wc_win_count": "Total Wins", "host_and_win": "Host & Win"
    }),
    caption="FIFA World Cup — All Tournaments 1930–2022"
))

In [ ]:
# Cell 10 — World Bank birth rate fetch (cache-first)
def fetch_world_bank_birth_rates(
    iso_codes: list,
    date_range: str = "1960:2024",
    cache_path: pathlib.Path = DATA_RAW
) -> pd.DataFrame:
    """
    Fetch crude birth rate (SP.DYN.CBRT.IN) from World Bank REST API.
    Caches to data/raw/wb_birth_rates.json to avoid repeated calls.
    Returns tidy DataFrame: [country, iso3, year, birth_rate]
    """
    cache_file = cache_path / "wb_birth_rates.json"

    if cache_file.exists():
        print(f"📦 Loading from cache: {cache_file.name}")
        with open(cache_file) as f:
            raw = json.load(f)
    else:
        iso_str = ";".join(iso_codes)
        url = (
            f"https://api.worldbank.org/v2/country/{iso_str}"
            f"/indicator/SP.DYN.CBRT.IN"
            f"?format=json&date={date_range}&per_page=2000"
        )
        print(f"🌐 Fetching from World Bank API...")
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        raw = resp.json()
        with open(cache_file, "w") as f:
            json.dump(raw, f)
        print(f"💾 Cached → {cache_file.name}")

    metadata, records = raw[0], raw[1]
    print(f"   Records: {metadata['total']}  |  Pages: {metadata['pages']}")

    rows = []
    for r in records:
        if r["value"] is not None:
            rows.append({
                "country":    r["country"]["value"],
                "iso3":       r["countryiso3code"],
                "year":       int(r["date"]),
                "birth_rate": float(r["value"]),
            })

    return pd.DataFrame(rows).sort_values(["iso3", "year"]).reset_index(drop=True)

print("fetch_world_bank_birth_rates() defined.")

In [ ]:
# Cell 11 — Load birth rate data
ALL_ISOS = [
    # Winners
    "URY", "ITA", "DEU", "BRA", "GBR", "ARG", "FRA", "ESP",
    # Non-winning hosts
    "CHE", "SWE", "CHL", "MEX", "USA",
    "JPN", "KOR", "ZAF", "RUS", "QAT",
]

df_birth = fetch_world_bank_birth_rates(ALL_ISOS)

print(f"\nBirth rate data: {df_birth.shape[0]} rows, {df_birth['iso3'].nunique()} countries")
print(f"Year range: {df_birth['year'].min()} – {df_birth['year'].max()}")
display(df_birth.head(8))

In [ ]:
# Cell 12 — Load happiness data (Our World in Data)
def fetch_happiness_data(cache_path: pathlib.Path = DATA_RAW) -> pd.DataFrame:
    """
    Fetch Cantril Ladder happiness scores from Our World in Data.
    Caches to data/raw/owid_happiness.csv.
    """
    cache_file = cache_path / "owid_happiness.csv"
    OWID_URL = "https://ourworldindata.org/grapher/happiness-cantril-ladder.csv?tab=chart"

    if cache_file.exists():
        print(f"📦 Loading from cache: {cache_file.name}")
        df = pd.read_csv(cache_file)
    else:
        print("🌐 Fetching happiness data from Our World in Data...")
        try:
            df = pd.read_csv(OWID_URL)
            df.to_csv(cache_file, index=False)
            print(f"💾 Cached → {cache_file.name}")
        except Exception as e:
            print(f"⚠️  Could not fetch OWID data: {e}")
            print("   Happiness correlation section will be skipped.")
            return pd.DataFrame(columns=["country", "iso3", "year", "happiness_score"])

    # Normalise column names (OWID format varies slightly by download date)
    rename_map = {}
    for col in df.columns:
        cl = col.lower()
        if "entity" in cl or cl == "country":
            rename_map[col] = "country"
        elif cl == "code" or cl == "iso":
            rename_map[col] = "iso3"
        elif cl == "year":
            rename_map[col] = "year"
        elif "satisfaction" in cl or "ladder" in cl or "happiness" in cl:
            rename_map[col] = "happiness_score"

    df = df.rename(columns=rename_map)
    df["happiness_score"] = pd.to_numeric(df.get("happiness_score", pd.NA), errors="coerce")
    df = df.dropna(subset=["happiness_score"])
    return df[["country", "iso3", "year", "happiness_score"]].copy()

df_happy = fetch_happiness_data()
HAPPINESS_AVAILABLE = len(df_happy) > 0
print(f"\nHappiness data: {len(df_happy)} rows, AVAILABLE={HAPPINESS_AVAILABLE}")
if HAPPINESS_AVAILABLE:
    print(f"Year range: {df_happy['year'].min()} – {df_happy['year'].max()}")

In [ ]:
# Cell 13 — Country name / ISO normalisation
ISO_TO_DISPLAY = {
    "URY": "Uruguay",      "ITA": "Italy",        "DEU": "Germany",
    "BRA": "Brazil",       "GBR": "England (UK)", "ARG": "Argentina",
    "FRA": "France",       "ESP": "Spain",
    "CHE": "Switzerland",  "SWE": "Sweden",       "CHL": "Chile",
    "MEX": "Mexico",       "USA": "United States", "JPN": "Japan",
    "KOR": "South Korea",  "ZAF": "South Africa",  "RUS": "Russia",
    "QAT": "Qatar",
}

FLAG_MAP = {
    "URY": "🇺🇾", "ITA": "🇮🇹", "DEU": "🇩🇪",
    "BRA": "🇧🇷", "GBR": "🏴",  "ARG": "🇦🇷",
    "FRA": "🇫🇷", "ESP": "🇪🇸",
    "CHE": "🇨🇭", "SWE": "🇸🇪", "CHL": "🇨🇱",
    "MEX": "🇲🇽", "USA": "🇺🇸", "JPN": "🇯🇵",
    "KOR": "🇰🇷", "ZAF": "🇿🇦", "RUS": "🇷🇺",
    "QAT": "🇶🇦",
}

df_birth["country_display"] = df_birth["iso3"].map(ISO_TO_DISPLAY)

# Happiness: map OWID country names → ISO3
OWID_TO_ISO = {
    "Germany": "DEU", "France": "FRA", "Argentina": "ARG",
    "Brazil": "BRA", "Spain": "ESP", "Italy": "ITA",
    "United Kingdom": "GBR", "Uruguay": "URY", "United States": "USA",
    "Japan": "JPN", "South Korea": "KOR", "South Africa": "ZAF",
    "Russia": "RUS", "Qatar": "QAT", "Chile": "CHL",
    "Mexico": "MEX", "Sweden": "SWE", "Switzerland": "CHE",
}
if HAPPINESS_AVAILABLE:
    df_happy["iso3_norm"] = df_happy["country"].map(OWID_TO_ISO)
    # Also try the iso3 column directly if present
    if "iso3" in df_happy.columns:
        mask = df_happy["iso3_norm"].isna()
        df_happy.loc[mask, "iso3_norm"] = df_happy.loc[mask, "iso3"]

print("✅ Normalisation complete.")
print(f"   Birth rate countries with display names: {df_birth['country_display'].notna().sum()} / {len(df_birth)}")

In [ ]:
# Cell 14 — Data coverage summary
coverage_rows = []
for iso in ALL_ISOS:
    sub = df_birth[df_birth["iso3"] == iso]
    coverage_rows.append({
        "Country": ISO_TO_DISPLAY.get(iso, iso),
        "ISO": iso,
        "First Year": int(sub["year"].min()) if len(sub) else "—",
        "Last Year":  int(sub["year"].max()) if len(sub) else "—",
        "Years Available": len(sub),
        "Coverage %": f"{len(sub)/65*100:.0f}%",
    })

df_cov = pd.DataFrame(coverage_rows).sort_values("Years Available", ascending=False)
display(style_wc_table(df_cov.reset_index(drop=True), caption="World Bank Data Coverage (1960–2024)"))

# Flag any critical missing data
print("\n⚠️  Missing data warnings:")
found_issues = False
for _, row in df_winners.iterrows():
    wy  = row["year"]
    iso = row["winner_iso"]
    needed = list(range(wy - 5, wy)) + [wy + 1]
    avail  = set(df_birth[df_birth["iso3"] == iso]["year"].tolist())
    missing = [y for y in needed if y not in avail]
    if missing:
        print(f"  {wy} {row['winner']:14s}: missing {missing}")
        found_issues = True
if not found_issues:
    print("  None — all required years are present.")

<div style="background: linear-gradient(90deg, #1a6b3c, #0d4a28);
            border-left: 6px solid #FFD700; border-radius: 10px;
            padding: 18px 28px; margin: 28px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <h2 style="color: #FFD700; margin: 0; font-size: 1.55em; font-weight: 700; letter-spacing: 1px;">
    ⚽ &nbsp; Section 3 &nbsp;|&nbsp; The World Cup Baby Boom
  </h2>
</div>

In [ ]:
# Cell 16 — Baseline computation helper
def compute_baseline(df_birth: pd.DataFrame, iso: str, win_year: int,
                     window: int = 5, min_years: int = 3):
    """Mean birth rate over [win_year-window, win_year-1]. Returns (mean, n_years)."""
    years = range(win_year - window, win_year)
    sub   = df_birth[(df_birth["iso3"] == iso) & (df_birth["year"].isin(years))]
    n     = len(sub)
    if n < min_years:
        return None, n
    return float(sub["birth_rate"].mean()), n

# Quick sanity check
bl, n = compute_baseline(df_birth, "BRA", 1994)
print(f"Brazil 1994 — 5yr baseline: {bl:.3f} per 1,000 (from {n} years)")

In [ ]:
# Cell 17 — Compute birth rate delta for all WC winners
results = []

for _, row in df_winners.iterrows():
    wy  = row["year"]
    iso = row["winner_iso"]

    # Post-win rate (Y+1)
    post_data = df_birth[(df_birth["iso3"] == iso) & (df_birth["year"] == wy + 1)]
    post_rate = float(post_data["birth_rate"].iloc[0]) if not post_data.empty else None

    # Win-year rate (Y)
    win_data  = df_birth[(df_birth["iso3"] == iso) & (df_birth["year"] == wy)]
    win_rate  = float(win_data["birth_rate"].iloc[0]) if not win_data.empty else None

    # Baseline
    baseline, n_bl = compute_baseline(df_birth, iso, wy)

    # Delta
    if post_rate is not None and baseline is not None:
        delta   = post_rate - baseline
        pct_chg = delta / baseline * 100
        ok      = True
    else:
        delta   = None
        pct_chg = None
        ok      = False

    # Years since last win
    iso_wins   = df_winners[df_winners["winner_iso"] == iso].sort_values("year")
    prior_wins = iso_wins[iso_wins["year"] < wy]
    years_since = int(wy - prior_wins["year"].iloc[-1]) if len(prior_wins) > 0 else None

    results.append({
        "year":             wy,
        "winner":           row["winner"],
        "winner_display":   row["winner_display"],
        "winner_iso":       iso,
        "host_and_win":     row["host_and_win"],
        "win_rate":         win_rate,
        "post_win_rate":    post_rate,
        "baseline_5yr":     baseline,
        "n_baseline_years": n_bl,
        "birth_rate_delta": delta,
        "birth_rate_pct":   pct_chg,
        "wc_win_count":     int(row["wc_win_count"]),
        "years_since_win":  years_since,
        "has_full_data":    ok,
    })

df_results     = pd.DataFrame(results)
df_valid       = df_results[df_results["has_full_data"]].copy().reset_index(drop=True)

print(f"Total WC editions:          {len(df_results)}")
print(f"With full data (analysed):  {len(df_valid)}")
print(f"Excluded (data gaps):       {len(df_results) - len(df_valid)}")

In [ ]:
# Cell 18 — Engineered features & labels
df_valid["direction"] = df_valid["birth_rate_delta"].apply(
    lambda x: "Positive ▲" if x > 0 else "Negative ▼"
)
df_valid["flag"]  = df_valid["winner_iso"].map(FLAG_MAP).fillna("")
df_valid["label"] = df_valid["flag"] + " " + df_valid["winner_display"]
df_valid["label_year"] = df_valid["label"] + " (" + df_valid["year"].astype(str) + ")"

print("Engineered features added. Preview:")
display(df_valid[["year", "label", "baseline_5yr", "win_rate", "post_win_rate",
                  "birth_rate_delta", "birth_rate_pct", "direction"]].head(8))

In [ ]:
# Cell 19 — Styled results table
def color_delta(val):
    if pd.isna(val): return ""
    color = WC_GREEN if val > 0 else "#8b0000"
    return f"color: {color}; font-weight: bold"

display_cols = {
    "year": "Year", "label": "Winner",
    "baseline_5yr": "Baseline (5yr avg)",
    "post_win_rate": "Post-Win Rate (Y+1)",
    "birth_rate_delta": "Δ Birth Rate",
    "birth_rate_pct": "% Change",
    "host_and_win": "Host & Win",
    "direction": "Direction",
}

styled_results = (
    df_valid[list(display_cols.keys())]
    .rename(columns=display_cols)
    .reset_index(drop=True)
    .style
    .map(color_delta, subset=["Δ Birth Rate"])
    .format({
        "Baseline (5yr avg)": "{:.3f}",
        "Post-Win Rate (Y+1)": "{:.3f}",
        "Δ Birth Rate": "{:+.3f}",
        "% Change": "{:+.2f}%",
    })
    .set_table_styles([
        {"selector": "table",
         "props": [("border-radius", "10px"), ("overflow", "hidden"),
                   ("width", "100%"), ("font-family", "'Helvetica Neue', Arial, sans-serif")]},
        {"selector": "thead th",
         "props": [("background-color", WC_GREEN), ("color", WC_GOLD),
                   ("font-weight", "700"), ("padding", "10px 14px"),
                   ("border-bottom", f"2px solid {WC_GOLD}")]},
        {"selector": "tbody td",
         "props": [("padding", "8px 14px"), ("border-bottom", "1px solid #c8e6c9")]},
        {"selector": "tbody tr:nth-child(even)",
         "props": [("background-color", WC_SAGE)]},
    ])
    .set_caption("Birth Rate Delta for All World Cup Winners (post-1960)")
    .hide(axis="index")
)
display(styled_results)

In [ ]:
# Cell 20 — Statistical summary
valid_deltas = df_valid["birth_rate_delta"].dropna()
n_pos = (valid_deltas > 0).sum()
n_neg = (valid_deltas < 0).sum()

t_stat, p_val = stats.ttest_1samp(valid_deltas, 0)

summary_html = f"""
<div style="background: linear-gradient(135deg, #0d4a28, #1a6b3c);
            border: 2px solid {WC_GOLD}; border-radius: 12px;
            padding: 24px 32px; margin: 20px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif; color: white;">
  <h3 style="color: {WC_GOLD}; margin: 0 0 16px 0; font-size: 1.3em;">📊 Statistical Summary</h3>
  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 12px;">
    <div><strong>Tournaments analysed:</strong> {len(valid_deltas)}</div>
    <div><strong>Positive delta (bump):</strong> {n_pos} ({n_pos/len(valid_deltas)*100:.0f}%)</div>
    <div><strong>Negative delta (drop):</strong> {n_neg} ({n_neg/len(valid_deltas)*100:.0f}%)</div>
    <div><strong>Mean delta:</strong> {valid_deltas.mean():+.3f} per 1,000</div>
    <div><strong>Median delta:</strong> {valid_deltas.median():+.3f} per 1,000</div>
    <div><strong>Std deviation:</strong> {valid_deltas.std():.3f}</div>
    <div><strong>Largest bump:</strong> {valid_deltas.max():+.3f}</div>
    <div><strong>Largest drop:</strong> {valid_deltas.min():+.3f}</div>
  </div>
  <hr style="border-color: {WC_GOLD}; opacity: 0.4; margin: 16px 0;">
  <div>
    <strong>One-sample t-test (H₀: mean delta = 0):</strong><br>
    t = {t_stat:.3f} &nbsp;|&nbsp; p = {p_val:.4f} &nbsp;
    {'<span style="color:#90EE90">✓ Significant (p&lt;0.05)</span>' if p_val < 0.05
     else '<span style="color:#FFB347">✗ Not significant (p≥0.05)</span>'}
  </div>
</div>
"""
display(HTML(summary_html))

<div style="background: linear-gradient(90deg, #1a6b3c, #0d4a28);
            border-left: 6px solid #FFD700; border-radius: 10px;
            padding: 18px 28px; margin: 28px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <h2 style="color: #FFD700; margin: 0; font-size: 1.55em; font-weight: 700; letter-spacing: 1px;">
    📈 &nbsp; Section 3c &nbsp;|&nbsp; Trend-Adjusted Analysis
  </h2>
  <p style="color: #e8f5e0; margin: 10px 0 0 0; font-size: 0.95em;">
    Isolating the World Cup effect from the global birth rate decline
  </p>
</div>

## Why Trend-Adjust?

The simple delta $\Delta BR_i = BR_{i,Y+1} - \overline{BR}_{i,Y-5:Y-1}$ compares the post-win year against a **flat** 5-year mean. But birth rates have been falling continuously since the 1960s. A country declining at −0.8 per year will almost always post a negative raw delta — not because the World Cup suppressed births, but because the downward trend continued.

**A better question:** does the winning year land **above or below where the pre-win trend would have taken us anyway?**

**Method:** For each WC win by country $i$ in year $Y$, fit a linear regression to birth rates in $[Y-5,\; Y-1]$:

$$\hat{BR}_{i,Y+1} = \hat{\beta}_0 + \hat{\beta}_1 \cdot (Y+1)$$

Then compute the **trend-adjusted delta**:

$$\Delta BR^{\text{trend}}_i = BR_{i,Y+1} - \hat{BR}_{i,Y+1}$$

- **Positive** → birth rate in Y+1 *exceeded* the expected trend (WC lifted births relative to trajectory)
- **Negative** → birth rate fell *faster* than the trend would predict
- **Near-zero** → WC had no discernible effect on trajectory

In [ ]:
# Cell 3c-1 — Compute trend-adjusted birth rate delta for all WC winners
# Fit linear regression to [Y-5, Y-1] birth rates, extrapolate to Y+1

def compute_trend_adjusted_delta(df_b, iso, win_year, window=5, min_years=3):
    """
    Fit a linear trend to birth rates in [win_year-window, win_year-1].
    Returns (trend_delta, slope, r_squared) or (None, None, None) if insufficient data.
    """
    sub = df_b[df_b["iso3"] == iso].copy()
    base_years = list(range(win_year - window, win_year))
    base = sub[sub["year"].isin(base_years)].dropna(subset=["birth_rate"])

    if len(base) < min_years:
        return None, None, None

    post_row = sub[sub["year"] == win_year + 1].dropna(subset=["birth_rate"])
    if post_row.empty:
        return None, None, None

    actual = float(post_row["birth_rate"].iloc[0])
    slope, intercept, r, _, _ = stats.linregress(base["year"].values, base["birth_rate"].values)
    predicted = slope * (win_year + 1) + intercept

    return actual - predicted, slope, r ** 2


trend_rows = []
for _, row in df_winners.iterrows():
    iso      = row["winner_iso"]
    win_year = row["year"]

    sub  = df_birth[df_birth["iso3"] == iso]
    post = sub[sub["year"] == win_year + 1]
    if post.empty:
        continue
    actual_br = float(post["birth_rate"].iloc[0])

    base_mean, n_base = compute_baseline(df_birth, iso, win_year)
    if base_mean is None:
        continue

    raw_delta = actual_br - base_mean

    tdelta, slope, r2 = compute_trend_adjusted_delta(df_birth, iso, win_year)
    if tdelta is None:
        continue

    trend_rows.append({
        "year":          win_year,
        "country":       ISO_TO_DISPLAY.get(iso, iso),
        "iso3":          iso,
        "actual_br":     actual_br,
        "baseline_mean": base_mean,
        "raw_delta":     raw_delta,
        "trend_slope":   slope,
        "trend_r2":      r2,
        "trend_delta":   tdelta,
        "trend_pct":     tdelta / actual_br * 100,
        "direction":     "Positive ▲" if tdelta > 0 else "Negative ▼",
        "host_and_win":  row.get("host_and_win", False),
    })

df_trend = pd.DataFrame(trend_rows).sort_values("trend_delta", ascending=False).reset_index(drop=True)

n_pos   = (df_trend["trend_delta"] > 0).sum()
n_neg   = (df_trend["trend_delta"] < 0).sum()
mean_td = df_trend["trend_delta"].mean()
t_stat, p_val = stats.ttest_1samp(df_trend["trend_delta"].dropna(), 0)

print(f"Trend-adjusted analysis — {len(df_trend)} WC editions with full data")
print(f"  Positive trend-delta (above trend): {n_pos}  ({n_pos/len(df_trend)*100:.0f}%)")
print(f"  Negative trend-delta (below trend): {n_neg}  ({n_neg/len(df_trend)*100:.0f}%)")
print(f"  Mean trend-adjusted delta: {mean_td:+.3f} births/1,000")
print(f"  One-sample t-test vs H₀=0: t={t_stat:.3f}, p={p_val:.4f}")
print()
print("Country-by-country breakdown (sorted by trend delta):")
FLAGS = {"BRA":"🇧🇷","DEU":"🇩🇪","ITA":"🇮🇹","ARG":"🇦🇷","FRA":"🇫🇷",
         "URY":"🇺🇾","GBR":"🇬🇧","ESP":"🇪🇸"}
for _, r in df_trend.iterrows():
    flag = FLAGS.get(r["iso3"], "🏳️")
    bar  = "▲" if r["trend_delta"] > 0 else "▼"
    print(f"  {flag} {r['country']:<12} {r['year']}  slope={r['trend_slope']:+.3f}/yr  "
          f"raw_Δ={r['raw_delta']:+.3f}  trend_Δ={r['trend_delta']:+.3f}  ({bar})")

In [ ]:
# Cell 3c-2 — Viz 9: Raw delta vs trend-adjusted delta comparison
# Side-by-side horizontal bars reveal where the flat-mean baseline misleads

df_plot9 = df_trend.copy()
df_plot9["label"] = df_plot9.apply(
    lambda r: f"{r['country']} '{str(r['year'])[2:]}", axis=1
)
df_plot9 = df_plot9.sort_values("trend_delta", ascending=True).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7), facecolor=WC_DARK)
fig.suptitle(
    "⚽  Raw Delta vs Trend-Adjusted Delta\n"
    "Left: BR[Y+1] − 5-yr mean  |  Right: BR[Y+1] − linear trend extrapolation",
    color=WC_GOLD, fontsize=14, fontweight="bold", y=1.01
)

for ax_i, (col, title) in enumerate([
    ("raw_delta",   "Raw Delta\n(vs flat 5-yr mean)"),
    ("trend_delta", "Trend-Adjusted Delta\n(vs linear trend extrapolated to Y+1)"),
]):
    ax = axes[ax_i]
    ax.set_facecolor(WC_DARK)

    vals   = df_plot9[col].values
    colors = [WC_GOLD if v > 0 else WC_RED for v in vals]
    bars   = ax.barh(df_plot9["label"], vals, color=colors, edgecolor="none", height=0.65)

    ax.axvline(0, color="white", linewidth=1.0, alpha=0.7)
    ax.set_title(title, color=WC_GOLD, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Births per 1,000 population", color="white", fontsize=10)
    ax.tick_params(colors="white", labelsize=9)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(axis="x", color="#2d8c55", alpha=0.35, linestyle="--")
    ax.set_yticklabels(df_plot9["label"], color="white", fontsize=9)

    # Value labels
    for bar, val in zip(bars, vals):
        x_off = 0.03 if val >= 0 else -0.03
        ha    = "left" if val >= 0 else "right"
        ax.text(val + x_off, bar.get_y() + bar.get_height() / 2,
                f"{val:+.2f}", va="center", ha=ha, color="white", fontsize=7.5)

    # Annotation for positive-count change
    n_pos_raw  = (df_plot9["raw_delta"]   > 0).sum()
    n_pos_tad  = (df_plot9["trend_delta"] > 0).sum()
    count = n_pos_raw if col == "raw_delta" else n_pos_tad
    ax.text(
        0.98, 0.02, f"{count}/{len(df_plot9)} positive",
        transform=ax.transAxes, color=WC_GOLD, fontsize=10,
        ha="right", va="bottom", fontweight="bold"
    )

plt.tight_layout()
plt.savefig(ASSETS / "viz9_trend_comparison.png", dpi=150, bbox_inches="tight",
            facecolor=WC_DARK)
plt.show()
print(f"✅ Saved viz9_trend_comparison.png")

In [ ]:
# Cell 3c-3 — Viz 10: Per-country trend lines + WC win markers
# Shows the birth rate time series, the pre-win linear trend extrapolated,
# and exactly where Y+1 landed (above or below trend)

TREND_SHOWCASE = [
    ("BRA", "Brazil",    [1958, 1962, 1970, 1994, 2002]),
    ("DEU", "Germany",   [1954, 1974, 1990, 2014]),
    ("ITA", "Italy",     [1934, 1938, 1982, 2006]),
    ("FRA", "France",    [1998, 2018]),
    ("ARG", "Argentina", [1978, 1986, 2022]),
    ("ESP", "Spain",     [2010]),
]

COUNTRY_COLORS = {
    "BRA": "#FFCC00", "DEU": "#3a8fce", "ITA": "#009246",
    "FRA": "#4169E1", "ARG": "#74ACDF", "ESP": "#AA151B",
}

fig, axes = plt.subplots(2, 3, figsize=(20, 11), facecolor=WC_DARK)
fig.suptitle(
    "⚽  Birth Rate Trend vs Actual — How Each World Cup Win Altered Trajectory\n"
    "Dashed line = linear trend extrapolated from 5 pre-win years  |  ★ = WC win year  |  ● = Y+1 actual",
    color=WC_GOLD, fontsize=13, fontweight="bold", y=1.01
)

FLAGS_V10 = {"BRA":"🇧🇷","DEU":"🇩🇪","ITA":"🇮🇹","FRA":"🇫🇷","ARG":"🇦🇷","ESP":"🇪🇸"}

for ax, (iso, name, win_years) in zip(axes.flatten(), TREND_SHOWCASE):
    ax.set_facecolor(WC_DARK)
    color = COUNTRY_COLORS.get(iso, WC_GOLD)

    sub = df_birth[df_birth["iso3"] == iso].dropna(subset=["birth_rate"])
    sub = sub.sort_values("year")

    # Full birth rate line
    ax.plot(sub["year"], sub["birth_rate"], color=color, linewidth=2.0,
            alpha=0.85, zorder=3)

    for wy in win_years:
        base_years = list(range(wy - 5, wy))
        base = sub[sub["year"].isin(base_years)].dropna(subset=["birth_rate"])
        post = sub[sub["year"] == wy + 1]

        if len(base) < 3 or post.empty:
            continue

        slope, intercept, r, _, _ = stats.linregress(
            base["year"].values, base["birth_rate"].values
        )

        # Draw trend line from Y-5 through Y+2
        t_years = np.arange(wy - 5, wy + 3)
        t_vals  = slope * t_years + intercept
        ax.plot(t_years, t_vals, color="white", linewidth=1.2, linestyle="--",
                alpha=0.5, zorder=2)

        # Gold star at win year
        win_row = sub[sub["year"] == wy]
        if not win_row.empty:
            ax.scatter(wy, float(win_row["birth_rate"].iloc[0]),
                       marker="*", s=200, color=WC_GOLD, zorder=6, linewidths=0)

        # Circle at Y+1: gold if above trend, red if below
        actual    = float(post["birth_rate"].iloc[0])
        predicted = slope * (wy + 1) + intercept
        dot_color = WC_GOLD if actual > predicted else WC_RED
        ax.scatter(wy + 1, actual, s=100, color=dot_color,
                   zorder=7, edgecolors="white", linewidths=1.0)

        # Vertical dotted line showing the gap
        ax.plot([wy + 1, wy + 1], [predicted, actual],
                color=dot_color, linewidth=1.8, linestyle=":", alpha=0.8, zorder=5)

    flag = FLAGS_V10.get(iso, "")
    ax.set_title(f"{flag}  {name}", color=color, fontsize=12, fontweight="bold")
    ax.set_ylabel("Birth Rate (per 1,000)", color="white", fontsize=9)
    ax.set_xlabel("Year", color="white", fontsize=9)
    ax.tick_params(colors="white", labelsize=8)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(color="#2d8c55", alpha=0.25, linestyle="--")

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color=WC_GOLD, marker="*", markersize=11, linestyle="None", label="WC win year"),
    Line2D([0], [0], color=WC_GOLD, marker="o", markersize=8,  linestyle="None", label="Y+1 above trend ▲"),
    Line2D([0], [0], color=WC_RED,  marker="o", markersize=8,  linestyle="None", label="Y+1 below trend ▼"),
    Line2D([0], [0], color="white", linewidth=1.2, linestyle="--", alpha=0.6, label="Pre-win trend extrapolated"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=4,
           facecolor=WC_DARK, edgecolor=WC_GOLD, labelcolor="white",
           fontsize=10, bbox_to_anchor=(0.5, -0.03))

plt.tight_layout()
plt.savefig(ASSETS / "viz10_trend_lines.png", dpi=150, bbox_inches="tight",
            facecolor=WC_DARK)
plt.show()
print("✅ Saved viz10_trend_lines.png")

<div style="background: linear-gradient(90deg, #1a6b3c, #0d4a28);
            border-left: 6px solid #FFD700; border-radius: 10px;
            padding: 18px 28px; margin: 28px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <h2 style="color: #FFD700; margin: 0; font-size: 1.55em; font-weight: 700; letter-spacing: 1px;">
    ⚽ &nbsp; Section 3b &nbsp;|&nbsp; Deep Dive — Monthly Births (European Winners)
  </h2>
  <p style="color: #e8f5e0; margin: 10px 0 0 0; font-size: 0.95em;">
    Eurostat <code>demo_fmonth</code> dataset &nbsp;·&nbsp; monthly live births, 1960–present
  </p>
</div>

## Why Monthly Data?

The annual analysis uses crude birth rate in **year Y+1** as a proxy for the 9-month lag — but this averaging hides the signal. World Cup finals fall in **late June / early July**. If there is a baby boom effect, conceptions spike in **July–August of Y**, producing births in **March–May of Y+1**. Annual data spreads these extra births across 12 months, diluting any bump.

Monthly data lets us ask the sharper question: **do births specifically in March–May of Y+1 exceed what we'd expect from seasonal patterns in baseline years?**

**Countries analysed:** European WC winners with Eurostat monthly data available — Germany (1990, 2014), France (1998, 2018), Italy (2006), Spain (2010).

**Method:**
$$\Delta_m = B_{i,\,Y+1,\,m} - \overline{B}_{i,\,[Y-5:Y-1],\,m}$$

where $B_{i,Y,m}$ = live births in country $i$, year $Y$, month $m$, and the baseline is the average of the same month across the 5 prior years.

In [ ]:
# Cell 3b-1 — Fetch Eurostat monthly live births (demo_fmonth)
# Eurostat demo_fmonth: annual freq, month dimension (M01-M12), year as time dimension

EUROSTAT_GEOS = ["DE", "FR", "IT", "ES"]

ISO2_TO_ISO3 = {"DE": "DEU", "FR": "FRA", "IT": "ITA", "ES": "ESP"}
ISO2_TO_NAME = {"DE": "Germany", "FR": "France", "IT": "Italy", "ES": "Spain"}
ISO2_FLAG    = {"DE": "🇩🇪", "FR": "🇫🇷", "IT": "🇮🇹", "ES": "🇪🇸"}

def fetch_eurostat_monthly_births(
    geo_codes: list = EUROSTAT_GEOS,
    since: str = "1980",
    cache_path: pathlib.Path = DATA_RAW
) -> pd.DataFrame:
    """
    Fetch monthly live births from Eurostat demo_fmonth.
    Dimensions: [freq, unit, month(M01-M12), geo, time(year)]
    Caches to data/raw/eurostat_monthly_births.json.
    Returns tidy DataFrame: [geo2, iso3, country, year, month, births]
    """
    cache_file = cache_path / "eurostat_monthly_births.json"

    if cache_file.exists():
        print(f"📦 Loading from cache: {cache_file.name}")
        with open(cache_file) as f:
            data = json.load(f)
    else:
        geo_params = "&".join(f"geo={g}" for g in geo_codes)
        url = (
            "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
            f"demo_fmonth?format=JSON&lang=EN&unit=NR&{geo_params}"
            f"&sinceTimePeriod={since}"
        )
        print(f"🌐 Fetching Eurostat monthly births ({since}–present)...")
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        with open(cache_file, "w") as f:
            json.dump(data, f)
        print(f"💾 Cached → {cache_file.name}")

    # Dimension order: [freq, unit, month, geo, time]
    dim_ids = data["id"]
    sizes   = data["size"]
    values  = data["value"]   # dict of str(flat_index) → value

    month_cat = data["dimension"]["month"]["category"]
    geo_cat   = data["dimension"]["geo"]["category"]
    time_cat  = data["dimension"]["time"]["category"]

    # Build position → code maps
    month_pos_to_code = {v: k for k, v in month_cat["index"].items()}
    geo_pos_to_code   = {v: k for k, v in geo_cat["index"].items()}
    time_pos_to_year  = {v: k for k, v in time_cat["index"].items()}

    n_month = sizes[dim_ids.index("month")]
    n_geo   = sizes[dim_ids.index("geo")]
    n_time  = sizes[dim_ids.index("time")]

    # Skip freq (size 1) and unit (size 1) — they're always 0
    # Flat index = month_i * n_geo * n_time + geo_i * n_time + time_i

    rows = []
    for key_str, val in values.items():
        if val is None:
            continue
        idx     = int(key_str)
        month_i = (idx // (n_geo * n_time)) % n_month
        geo_i   = (idx // n_time) % n_geo
        time_i  =  idx % n_time

        month_code = month_pos_to_code.get(month_i, "")
        geo2       = geo_pos_to_code.get(geo_i, "")
        year_str   = time_pos_to_year.get(time_i, "")

        # Skip TOTAL and UNK months; keep only M01-M12
        if not month_code.startswith("M") or len(month_code) != 3:
            continue
        try:
            month_num = int(month_code[1:])
            year      = int(year_str)
        except ValueError:
            continue

        rows.append({
            "geo2":    geo2,
            "iso3":    ISO2_TO_ISO3.get(geo2, geo2),
            "country": ISO2_TO_NAME.get(geo2, geo2),
            "year":    year,
            "month":   month_num,
            "births":  float(val),
        })

    df = pd.DataFrame(rows).sort_values(["geo2", "year", "month"]).reset_index(drop=True)
    return df

df_monthly = fetch_eurostat_monthly_births()
print(f"\nMonthly births data: {len(df_monthly):,} rows")
print(f"Countries: {df_monthly['country'].unique().tolist()}")
print(f"Year range: {df_monthly['year'].min()} – {df_monthly['year'].max()}")
display(df_monthly.head(8))

In [ ]:
# Cell 3b-2 — Compute monthly birth delta for European WC wins
# For each win: births in each month of Y+1 vs. same month averaged over Y-5 to Y-1

EURO_WC_WINS = [
    # (iso2, iso3, name, win_year)
    ("DE", "DEU", "Germany",  1990),
    ("DE", "DEU", "Germany",  2014),
    ("FR", "FRA", "France",   1998),
    ("FR", "FRA", "France",   2018),
    ("IT", "ITA", "Italy",    2006),
    ("ES", "ESP", "Spain",    2010),
]

MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun",
               "Jul","Aug","Sep","Oct","Nov","Dec"]
# 9-month baby boom window: conceptions in Jul-Aug → births in Apr-May (months 4-5)
# We highlight months 3-6 (Mar, Apr, May, Jun) as the likely spike window
BOOM_MONTHS = [3, 4, 5, 6]

monthly_deltas = []

for iso2, iso3, name, wy in EURO_WC_WINS:
    sub = df_monthly[df_monthly["geo2"] == iso2].copy()
    if sub.empty:
        print(f"  ⚠️ No data for {name}")
        continue

    post_year = wy + 1
    base_years = list(range(wy - 5, wy))

    for m in range(1, 13):
        post_row = sub[(sub["year"] == post_year) & (sub["month"] == m)]
        base_rows = sub[(sub["year"].isin(base_years)) & (sub["month"] == m)]

        if post_row.empty or len(base_rows) < 3:
            continue

        post_val  = float(post_row["births"].iloc[0])
        base_mean = float(base_rows["births"].mean())
        delta     = post_val - base_mean
        pct       = delta / base_mean * 100

        monthly_deltas.append({
            "iso2":       iso2,
            "iso3":       iso3,
            "country":    name,
            "win_year":   wy,
            "month":      m,
            "month_name": MONTH_NAMES[m - 1],
            "post_births": post_val,
            "base_births": base_mean,
            "delta":       delta,
            "pct_change":  pct,
            "in_boom_window": m in BOOM_MONTHS,
        })

df_mdelta = pd.DataFrame(monthly_deltas)
print(f"Monthly delta rows: {len(df_mdelta)}")
print(f"\nBoom-window (Mar–Jun) mean deltas by win:")
boom = df_mdelta[df_mdelta["in_boom_window"]]
for (iso3, wy), g in boom.groupby(["iso3", "win_year"]):
    flag = ISO2_FLAG.get({v: k for k, v in ISO2_TO_ISO3.items()}.get(iso3, ""), "")
    print(f"  {flag} {g['country'].iloc[0]} {wy}: mean Δ = {g['delta'].mean():+.0f} births/month  "
          f"({g['pct_change'].mean():+.1f}%)")

In [ ]:
# Cell 3b-3 — Viz 7: Monthly birth delta bar charts (2×3 grid)
# Each panel = one WC win, bars = monthly delta (post-win Y+1 minus 5yr baseline)
# Gold bars = boom window (Mar–Jun), grey bars = other months

n_wins = len(EURO_WC_WINS)
ncols  = 3
nrows  = (n_wins + ncols - 1) // ncols   # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 5.5), facecolor=WC_DARK)
axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]

for ax_idx, (iso2, iso3, name, wy) in enumerate(EURO_WC_WINS):
    ax   = axes_flat[ax_idx]
    flag = ISO2_FLAG.get(iso2, "")
    sub  = df_mdelta[(df_mdelta["iso3"] == iso3) & (df_mdelta["win_year"] == wy)].sort_values("month")

    if sub.empty:
        ax.set_visible(False)
        continue

    bar_colors = [WC_GOLD if m in BOOM_MONTHS else "#4a7c59" for m in sub["month"]]
    bars = ax.bar(sub["month_name"], sub["delta"], color=bar_colors, edgecolor="none", width=0.7)

    # Value labels on non-zero bars
    for bar, val in zip(bars, sub["delta"]):
        if abs(val) > 50:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    val + (abs(val) * 0.04 if val >= 0 else -abs(val) * 0.04),
                    f"{val:+.0f}", ha="center",
                    va="bottom" if val >= 0 else "top",
                    color="white", fontsize=8)

    # Shade boom window background
    for m in BOOM_MONTHS:
        if m <= len(sub):
            ax.axvspan(m - 1.5, m - 0.5, alpha=0.07, color=WC_GOLD, zorder=0)

    ax.axhline(0, color="white", linewidth=0.8, alpha=0.5, linestyle="--")
    ax.set_facecolor(WC_DARK)
    ax.set_title(f"{flag} {name}  🏆 {wy}", color=WC_GOLD, fontsize=12, fontweight="bold")
    ax.set_ylabel("Δ Births vs. 5yr baseline", color="white", fontsize=9)
    ax.set_xlabel("Month of Y+1", color="white", fontsize=9)
    ax.tick_params(colors="white", labelsize=8)
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.grid(axis="y", color="#2d8c55", alpha=0.3, linestyle="--")

# Hide unused axes
for ax in axes_flat[n_wins:]:
    ax.set_visible(False)

# Legend
legend_patches = [
    mpatches.Patch(color=WC_GOLD,   label="Mar–Jun (9-month baby boom window)"),
    mpatches.Patch(color="#4a7c59", label="Other months"),
]
fig.legend(handles=legend_patches, facecolor=WC_DARK, edgecolor=WC_GOLD,
           labelcolor="white", loc="lower center", ncol=2,
           fontsize=10, bbox_to_anchor=(0.5, -0.02))

fig.suptitle(
    "⚽  Monthly Birth Difference After Winning the World Cup\n"
    "Post-Win Year (Y+1) minus 5-Year Monthly Baseline  |  European Winners",
    color=WC_GOLD, fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig(ASSETS / "viz7_monthly_delta.png", dpi=150, bbox_inches="tight", facecolor=WC_DARK)
plt.show()

In [ ]:
# Cell 3b-4 — Viz 8: Monthly births timeline — actual vs baseline ribbon
# Focus on Germany 2014 (strongest annual signal) and France 1998 (host+win)
# Shows raw monthly birth counts: post-win year (gold line) vs baseline band (grey ribbon)

DEEP_DIVE = [
    ("DE", "DEU", "Germany", 2014, "#FFCC00"),
    ("FR", "FRA", "France",  1998, "#4169E1"),
    ("DE", "DEU", "Germany", 1990, "#88cc88"),
    ("FR", "FRA", "France",  2018, "#87CEEB"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor=WC_DARK)

for ax, (iso2, iso3, name, wy, color) in zip(axes.flatten(), DEEP_DIVE):
    flag      = ISO2_FLAG.get(iso2, "")
    sub       = df_monthly[df_monthly["geo2"] == iso2].copy()
    post_year = wy + 1
    base_years = list(range(wy - 5, wy))

    post_data = sub[sub["year"] == post_year].sort_values("month")
    base_data = sub[sub["year"].isin(base_years)]

    if post_data.empty:
        ax.set_visible(False)
        continue

    base_stats = (
        base_data.groupby("month")["births"]
                 .agg(["mean", "std", "min", "max"])
                 .reset_index()
    )

    months = base_stats["month"].values
    base_mean = base_stats["mean"].values
    base_lo   = base_stats["mean"].values - base_stats["std"].values
    base_hi   = base_stats["mean"].values + base_stats["std"].values

    ax.set_facecolor(WC_DARK)

    # Baseline ribbon (±1 SD)
    ax.fill_between(months, base_lo, base_hi, alpha=0.20, color="white", label="Baseline ±1 SD")
    ax.plot(months, base_mean, color="white", linewidth=1.5, alpha=0.55,
            linestyle="--", label="Baseline mean")

    # Post-win year line
    post_months = post_data["month"].values
    post_births = post_data["births"].values
    ax.plot(post_months, post_births, color=color, linewidth=2.5, zorder=4,
            label=f"{post_year} (post-win)")

    # Shade the boom window (months 3-6)
    ax.axvspan(2.5, 6.5, alpha=0.10, color=WC_GOLD, zorder=0, label="Baby boom window")
    for m in BOOM_MONTHS:
        pr = post_data[post_data["month"] == m]
        if not pr.empty:
            ax.scatter(m, pr["births"].iloc[0], s=100, color=WC_GOLD,
                       zorder=5, edgecolors="white", linewidths=1.2)

    ax.set_facecolor(WC_DARK)
    ax.set_title(f"{flag} {name}  🏆 {wy}  →  births in {post_year}",
                 color=color, fontsize=12, fontweight="bold")
    ax.set_xlabel("Month", color="white", fontsize=10)
    ax.set_ylabel("Live Births", color="white", fontsize=10)
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(MONTH_NAMES, fontsize=8)
    ax.tick_params(colors="white")
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.grid(color="#2d8c55", alpha=0.3, linestyle="--")
    ax.legend(facecolor=WC_DARK, edgecolor=WC_GOLD, labelcolor="white",
              fontsize=8, loc="upper right")

fig.suptitle(
    "⚽  Monthly Live Births — Post-Win Year vs. 5-Year Baseline Band\n"
    "Gold shading = 9-month baby boom window (Mar–Jun)  |  ● = highlighted months",
    color=WC_GOLD, fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.savefig(ASSETS / "viz8_monthly_ribbon.png", dpi=150, bbox_inches="tight", facecolor=WC_DARK)
plt.show()

<div style="background: linear-gradient(90deg, #1a6b3c, #0d4a28);
            border-left: 6px solid #FFD700; border-radius: 10px;
            padding: 18px 28px; margin: 28px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <h2 style="color: #FFD700; margin: 0; font-size: 1.55em; font-weight: 700; letter-spacing: 1px;">
    ⚽ &nbsp; Section 4 &nbsp;|&nbsp; Visualisations
  </h2>
</div>

In [ ]:
# Cell 22 — Viz 1: Horizontal bar chart — birth rate delta by winner
df_plot = df_valid.sort_values("birth_rate_delta", ascending=True).reset_index(drop=True)
colors  = [WC_GOLD if d > 0 else WC_RED for d in df_plot["birth_rate_delta"]]
labels  = df_plot["label_year"].tolist()

fig, ax = plt.subplots(figsize=(13, max(8, len(df_plot) * 0.52)), facecolor=WC_DARK)
ax.set_facecolor(WC_DARK)

bars = ax.barh(labels, df_plot["birth_rate_delta"], color=colors,
               edgecolor="none", height=0.68)

# Value labels
for bar, val in zip(bars, df_plot["birth_rate_delta"]):
    offset = 0.05 if val >= 0 else -0.05
    ha     = "left"  if val >= 0 else "right"
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f"{val:+.3f}", va="center", ha=ha, color="white", fontsize=9)

ax.axvline(0, color="white", linewidth=1.2, alpha=0.6, linestyle="--")
ax.set_xlabel("Birth Rate Delta  (per 1,000 people)", color="white", fontsize=12, labelpad=10)
ax.set_title(
    "⚽  Birth Rate Change After Winning the World Cup\n"
    "Post-Win Year (Y+1) vs. 5-Year Pre-Win Average",
    color=WC_GOLD, fontsize=14, fontweight="bold", pad=16
)

for spine in ax.spines.values(): spine.set_visible(False)
ax.grid(axis="x", color="#2d8c55", alpha=0.3, linestyle="--")
ax.tick_params(colors="white", labelsize=10)

legend_patches = [
    mpatches.Patch(color=WC_GOLD, label="Birth rate increased ▲"),
    mpatches.Patch(color=WC_RED,  label="Birth rate decreased ▼"),
]
ax.legend(handles=legend_patches, facecolor=WC_DARK, edgecolor=WC_GOLD,
          labelcolor="white", loc="lower right", fontsize=10)

plt.tight_layout()
plt.savefig(ASSETS / "viz1_delta_bar.png", dpi=150, bbox_inches="tight", facecolor=WC_DARK)
plt.show()

In [ ]:
# Cell 23 — Viz 2: 2×2 birth rate timeline for key countries
HIGHLIGHT = [
    ("BRA", "Brazil",    "#009c3b"),
    ("DEU", "Germany",   "#FFCC00"),
    ("FRA", "France",    "#4169E1"),
    ("ARG", "Argentina", "#74ACDF"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor=WC_DARK)
fig.suptitle(
    "⚽  Birth Rate Trends — Major World Cup Winners\n"
    "(dashed line = win year  |  ● = post-win birth rate Y+1)",
    color=WC_GOLD, fontsize=14, fontweight="bold", y=1.01
)

for ax, (iso, name, color) in zip(axes.flatten(), HIGHLIGHT):
    sub   = df_birth[df_birth["iso3"] == iso].sort_values("year")
    wins  = df_winners[df_winners["winner_iso"] == iso]["year"].tolist()

    ax.set_facecolor(WC_DARK)
    ax.plot(sub["year"], sub["birth_rate"], color=color, linewidth=2.5, zorder=3)
    ax.fill_between(sub["year"], sub["birth_rate"], alpha=0.12, color=color)

    for wy in wins:
        # Vertical dashed marker at win year
        ax.axvline(wy, color=WC_GOLD, linewidth=1.0, alpha=0.65, linestyle="--", zorder=2)
        # Gold dot at Y+1
        post = sub[sub["year"] == wy + 1]
        if not post.empty:
            ax.scatter(wy + 1, post["birth_rate"].iloc[0],
                       s=110, zorder=5, color=WC_GOLD,
                       edgecolors="white", linewidths=1.5)
        # Small year label
        win_row = sub[sub["year"] == wy]
        if not win_row.empty:
            ax.annotate(
                f"🏆 {wy}",
                xy=(wy, win_row["birth_rate"].iloc[0]),
                xytext=(4, 8), textcoords="offset points",
                color=WC_GOLD, fontsize=8, fontweight="bold"
            )

    ax.set_title(f"{FLAG_MAP.get(iso, '')} {name}", color=color,
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Year", color="white", fontsize=10)
    ax.set_ylabel("Birth Rate (per 1,000)", color="white", fontsize=10)
    ax.tick_params(colors="white", labelsize=9)
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.grid(color="#2d8c55", alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig(ASSETS / "viz2_timeline.png", dpi=150, bbox_inches="tight", facecolor=WC_DARK)
plt.show()

In [ ]:
# Cell 24 — Viz 3: Birth rate heatmap across winning nations
WINNER_ISOS = ["BRA", "ARG", "DEU", "FRA", "ITA", "ESP", "GBR", "URY"]

df_heat = df_birth[
    (df_birth["iso3"].isin(WINNER_ISOS)) & (df_birth["year"] >= 1965)
].copy()
df_heat["label"] = df_heat["iso3"].map(
    lambda x: f"{FLAG_MAP.get(x, '')} {ISO_TO_DISPLAY.get(x, x)}"
)

pivot = df_heat.pivot(index="label", columns="year", values="birth_rate")
# Reorder rows by total win count (descending)
win_order = (
    df_winners[df_winners["winner_iso"].isin(WINNER_ISOS)]
    .groupby("winner_iso")["year"].count()
    .sort_values(ascending=False)
    .index.tolist()
)
ordered_labels = [
    f"{FLAG_MAP.get(iso, '')} {ISO_TO_DISPLAY.get(iso, iso)}" for iso in win_order
]
pivot = pivot.reindex([l for l in ordered_labels if l in pivot.index])

fig, ax = plt.subplots(figsize=(22, 5), facecolor=WC_DARK)
ax.set_facecolor(WC_DARK)

sns.heatmap(
    pivot, ax=ax,
    cmap="YlGn",
    linewidths=0.2, linecolor="#1a1a1a",
    cbar_kws={"label": "Birth Rate (per 1,000)", "shrink": 0.75},
)
ax.collections[0].colorbar.ax.yaxis.label.set_color("white")
ax.collections[0].colorbar.ax.tick_params(colors="white")

# Mark win years with gold stars
for _, wrow in df_winners[df_winners["winner_iso"].isin(WINNER_ISOS)].iterrows():
    wy  = wrow["year"]
    iso = wrow["winner_iso"]
    lbl = f"{FLAG_MAP.get(iso, '')} {ISO_TO_DISPLAY.get(iso, iso)}"
    if lbl in pivot.index and wy in pivot.columns:
        y_idx = list(pivot.index).index(lbl)
        x_idx = list(pivot.columns).index(wy)
        ax.scatter(x_idx + 0.5, y_idx + 0.5, marker="*",
                   s=280, color=WC_GOLD, zorder=6, edgecolors="black", linewidths=0.5)

ax.set_title(
    "Birth Rate Heatmap — World Cup Winning Nations (1965–2024)\n"
    "★ = World Cup win year",
    color=WC_GOLD, fontsize=13, fontweight="bold", pad=14
)
ax.tick_params(colors="white", labelsize=8)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(ASSETS / "viz3_heatmap.png", dpi=150, bbox_inches="tight", facecolor=WC_DARK)
plt.show()

In [ ]:
# Cell 25 — Viz 4: Plotly interactive — happiness vs. birth rate delta
# Build scatter data: all winners with happiness data available
scatter_rows = []
if HAPPINESS_AVAILABLE:
    for _, row in df_valid.iterrows():
        wy  = row["year"]
        iso = row["winner_iso"]
        hap = df_happy[
            (df_happy.get("iso3_norm", pd.Series()) == iso) & (df_happy["year"] == wy + 1)
        ]
        if not hap.empty:
            scatter_rows.append({
                "winner":           row["winner_display"],
                "flag":             FLAG_MAP.get(iso, ""),
                "year":             wy,
                "iso3":             iso,
                "birth_rate_delta": row["birth_rate_delta"],
                "happiness_score":  float(hap["happiness_score"].iloc[0]),
            })

df_scatter = pd.DataFrame(scatter_rows) if scatter_rows else pd.DataFrame()

if not df_scatter.empty:
    df_scatter["label"] = df_scatter["flag"] + " " + df_scatter["winner"] + " " + df_scatter["year"].astype(str)

    fig_plotly = px.scatter(
        df_scatter,
        x="happiness_score",
        y="birth_rate_delta",
        text="label",
        color_discrete_sequence=[WC_GOLD],
        title="⚽ Does National Happiness Predict the WC Baby Boom?",
        labels={
            "happiness_score":  "Happiness Score (Cantril Ladder, Y+1)",
            "birth_rate_delta": "Birth Rate Delta (per 1,000)",
        },
        template="plotly_dark",
    )
    fig_plotly.update_traces(
        textposition="top center",
        marker=dict(size=16, line=dict(color="white", width=1.5)),
    )
    fig_plotly.update_layout(
        plot_bgcolor=WC_DARK,
        paper_bgcolor=WC_DARK,
        title_font_color=WC_GOLD,
        font_color="white",
        title_font_size=16,
    )
    fig_plotly.add_hline(y=0, line_dash="dash", line_color="white", opacity=0.4)
    # Save static PNG for PDF
    try:
        fig_plotly.write_image(str(ASSETS / "viz4_scatter.png"), width=900, height=550)
    except Exception:
        pass  # kaleido may not be available; chart still shows interactively
    fig_plotly.show()

    display(HTML(f"""
    <div style="background: #fff3cd; border: 1px solid #ffc107; border-radius: 8px;
                padding: 12px 18px; margin: 12px 0; font-family: Arial, sans-serif;">
      <strong>⚠️ Note:</strong> Only {len(df_scatter)} post-2012 World Cup win(s) have
      overlapping happiness data. Correlation results are <em>exploratory only</em> —
      N={len(df_scatter)} is too small for statistical inference.
    </div>
    """))
else:
    print("ℹ️  No overlapping happiness data found for any WC winner. Skipping Viz 4.")

<div style="background: linear-gradient(90deg, #1a6b3c, #0d4a28);
            border-left: 6px solid #FFD700; border-radius: 10px;
            padding: 18px 28px; margin: 28px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <h2 style="color: #FFD700; margin: 0; font-size: 1.55em; font-weight: 700; letter-spacing: 1px;">
    ⚽ &nbsp; Section 5 &nbsp;|&nbsp; Host Country Effect
  </h2>
</div>

In [ ]:
# Cell 27 — Host country birth rate delta
host_results = []

for _, row in df_hosts.drop_duplicates(subset=["year", "host_iso"]).iterrows():
    wy  = row["year"]
    iso = row["host_iso"]

    post = df_birth[(df_birth["iso3"] == iso) & (df_birth["year"] == wy + 1)]
    bl, n_bl = compute_baseline(df_birth, iso, wy)

    if post.empty or bl is None:
        continue

    post_rate = float(post["birth_rate"].iloc[0])
    delta     = post_rate - bl
    pct       = delta / bl * 100

    host_results.append({
        "year":             wy,
        "host":             row["host"],
        "host_iso":         iso,
        "host_and_win":     row["host_and_win"],
        "flag":             FLAG_MAP.get(iso, ""),
        "baseline_5yr":     bl,
        "post_host_rate":   post_rate,
        "birth_rate_delta": delta,
        "birth_rate_pct":   pct,
    })

df_host = pd.DataFrame(host_results)
print(f"Host nations analysed: {len(df_host)}")

display(style_wc_table(
    df_host.assign(label=df_host["flag"] + " " + df_host["host"])
           [["year", "label", "baseline_5yr", "post_host_rate",
             "birth_rate_delta", "birth_rate_pct", "host_and_win"]]
    .rename(columns={
        "year": "Year", "label": "Host Nation",
        "baseline_5yr": "Baseline", "post_host_rate": "Post-Host Rate (Y+1)",
        "birth_rate_delta": "Δ Birth Rate", "birth_rate_pct": "% Change",
        "host_and_win": "Also Won",
    })
    .reset_index(drop=True),
    caption="Birth Rate Delta for World Cup Host Nations"
))

In [ ]:
# Cell 28 — Compare winner vs. host effects
winner_only = df_valid[~df_valid["host_and_win"]]["birth_rate_delta"]
host_only   = df_host[~df_host["host_and_win"]]["birth_rate_delta"]
host_win    = df_host[df_host["host_and_win"]]["birth_rate_delta"]
all_winners = df_valid["birth_rate_delta"]

print("Mean birth rate delta comparison")
print("=" * 45)
print(f"  Winners only (not host):  {winner_only.mean():+.3f}  (n={len(winner_only)})")
print(f"  Hosts only (not winning): {host_only.mean():+.3f}  (n={len(host_only)})")
print(f"  Host + Winner:            {host_win.mean():+.3f}  (n={len(host_win)})")
print(f"  All winners:              {all_winners.mean():+.3f}  (n={len(all_winners)})")

In [ ]:
# Cell 29 — Viz 5: Winner vs. Host effect bar chart
categories = ["Winners\n(not host)", "Hosts\n(not winning)", "Host\n+ Winner"]
means      = [winner_only.mean(), host_only.mean(), host_win.mean()]
stds       = [winner_only.std(), host_only.std(), host_win.std()]
counts     = [len(winner_only), len(host_only), len(host_win)]
bar_colors = [WC_GOLD, WC_BLUE, "#ff9f40"]

fig, ax = plt.subplots(figsize=(9, 6), facecolor=WC_DARK)
ax.set_facecolor(WC_DARK)

bars = ax.bar(
    categories, means,
    color=bar_colors, edgecolor="none", width=0.52,
    yerr=stds, capsize=6, error_kw={"color": "white", "linewidth": 1.5},
)

for bar, val, n in zip(bars, means, counts):
    ypos = val + (max(stds) * 0.15 if val >= 0 else min(stds) * 0.15)
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        ypos + 0.06,
        f"{val:+.3f}\n(n={n})",
        ha="center", va="bottom", color="white", fontsize=11, fontweight="bold"
    )

ax.axhline(0, color="white", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_ylabel("Mean Birth Rate Delta (per 1,000)  ±1 SD", color="white", fontsize=11)
ax.set_title(
    "⚽  World Cup Effect on Birth Rate:\nWinners vs. Host Nations",
    color=WC_GOLD, fontsize=13, fontweight="bold", pad=14
)
ax.tick_params(colors="white", labelsize=11)
for spine in ax.spines.values(): spine.set_visible(False)
ax.grid(axis="y", color="#2d8c55", alpha=0.3, linestyle="--")

plt.tight_layout()
plt.savefig(ASSETS / "viz5_host_vs_winner.png", dpi=150, bbox_inches="tight", facecolor=WC_DARK)
plt.show()

<div style="background: linear-gradient(90deg, #1a6b3c, #0d4a28);
            border-left: 6px solid #FFD700; border-radius: 10px;
            padding: 18px 28px; margin: 28px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <h2 style="color: #FFD700; margin: 0; font-size: 1.55em; font-weight: 700; letter-spacing: 1px;">
    ⚽ &nbsp; Section 6 &nbsp;|&nbsp; Happiness Index Correlation
  </h2>
</div>

In [ ]:
# Cell 31 — Merge birth rate delta with happiness scores
corr_rows = []

if HAPPINESS_AVAILABLE:
    for _, row in df_valid.iterrows():
        wy  = row["year"]
        iso = row["winner_iso"]
        hap = df_happy[
            (df_happy.get("iso3_norm", pd.Series(dtype=str)) == iso) &
            (df_happy["year"] == wy + 1)
        ]
        if not hap.empty:
            corr_rows.append({
                "winner":           row["winner_display"],
                "year":             wy,
                "iso3":             iso,
                "flag":             FLAG_MAP.get(iso, ""),
                "birth_rate_delta": row["birth_rate_delta"],
                "happiness_score":  float(hap["happiness_score"].iloc[0]),
                "host_and_win":     row["host_and_win"],
            })

df_corr = pd.DataFrame(corr_rows) if corr_rows else pd.DataFrame()
CORR_AVAILABLE = len(df_corr) >= 3

if not df_corr.empty:
    df_corr["label"] = df_corr["flag"] + " " + df_corr["winner"] + " " + df_corr["year"].astype(str)
    print(f"Correlation dataset: {len(df_corr)} data points")
    display(df_corr[["label", "birth_rate_delta", "happiness_score", "host_and_win"]])
else:
    print("ℹ️  No overlapping data for correlation analysis.")

In [ ]:
# Cell 32 — Correlation analysis
pearson_r = spearman_r = p_pearson = p_spearman = None

if CORR_AVAILABLE:
    pearson_r,  p_pearson  = stats.pearsonr(
        df_corr["happiness_score"], df_corr["birth_rate_delta"])
    spearman_r, p_spearman = stats.spearmanr(
        df_corr["happiness_score"], df_corr["birth_rate_delta"])

    strength = (
        "weak" if abs(pearson_r) < 0.3 else
        "moderate" if abs(pearson_r) < 0.6 else
        "strong"
    )
    direction = "positive" if pearson_r > 0 else "negative"

    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #0d4a28, #1a6b3c);
                border: 2px solid {WC_GOLD}; border-radius: 12px;
                padding: 20px 28px; margin: 16px 0;
                font-family: 'Helvetica Neue', Arial, sans-serif; color: white;">
      <h3 style="color: {WC_GOLD}; margin: 0 0 12px 0;">📈 Correlation Results</h3>
      <p><strong>Pearson r:</strong>  {pearson_r:+.3f} &nbsp;(p = {p_pearson:.4f})</p>
      <p><strong>Spearman ρ:</strong> {spearman_r:+.3f} &nbsp;(p = {p_spearman:.4f})</p>
      <p><strong>Interpretation:</strong> {strength.capitalize()} {direction} correlation.
         Happiness score {'does' if abs(pearson_r) >= 0.3 else 'does not'} appear to
         co-vary with the birth rate delta.</p>
      <p style="color: #ffd700; font-size: 0.9em;">⚠️ N={len(df_corr)} — results are
         exploratory only. Do not draw causal conclusions.</p>
    </div>
    """))
elif not df_corr.empty:
    print(f"Only {len(df_corr)} data point(s) — correlation requires at least 3. Skipping.")
else:
    print("No overlapping happiness data. Correlation skipped.")

In [ ]:
# Cell 33 — Viz 6: Happiness vs. birth rate delta scatter
if not df_corr.empty:
    fig, ax = plt.subplots(figsize=(9, 6), facecolor=WC_DARK)
    ax.set_facecolor(WC_DARK)

    ax.scatter(
        df_corr["happiness_score"], df_corr["birth_rate_delta"],
        s=200, color=WC_GOLD, edgecolors="white", linewidths=1.5, zorder=5
    )
    for _, r in df_corr.iterrows():
        ax.annotate(
            f"{r['flag']} {r['winner']} {r['year']}",
            xy=(r["happiness_score"], r["birth_rate_delta"]),
            xytext=(8, 5), textcoords="offset points",
            color="white", fontsize=10
        )

    if CORR_AVAILABLE and pearson_r is not None:
        m, b = np.polyfit(df_corr["happiness_score"], df_corr["birth_rate_delta"], 1)
        x_line = np.linspace(df_corr["happiness_score"].min() - 0.1,
                              df_corr["happiness_score"].max() + 0.1, 100)
        ax.plot(x_line, m * x_line + b, color=WC_BLUE, linewidth=1.8,
                linestyle="--", alpha=0.85, label=f"Trend (r = {pearson_r:+.2f})")
        ax.legend(facecolor=WC_DARK, edgecolor=WC_GOLD, labelcolor="white", fontsize=10)

    ax.axhline(0, color="white", linewidth=0.8, linestyle=":", alpha=0.45)
    ax.set_xlabel("Happiness Score (Cantril Ladder, Y+1)", color="white", fontsize=12)
    ax.set_ylabel("Birth Rate Delta (per 1,000)",          color="white", fontsize=12)
    ax.set_title(
        "⚽  Does National Happiness Predict the WC Baby Boom?\n"
        "(Post-2012 World Cup Winners)",
        color=WC_GOLD, fontsize=13, fontweight="bold", pad=14
    )
    ax.tick_params(colors="white")
    for spine in ax.spines.values(): spine.set_visible(False)
    ax.grid(color="#2d8c55", alpha=0.25)

    plt.tight_layout()
    plt.savefig(ASSETS / "viz6_happiness_scatter.png", dpi=150, bbox_inches="tight", facecolor=WC_DARK)
    plt.show()
else:
    print("ℹ️  No data for happiness scatter. Skipping Viz 6.")

## Interpreting the Happiness Correlation

The World Happiness Report provides Cantril Ladder scores (0–10) from Gallup World Poll data. We look at the score in **year Y+1** — the same year in which the birth rate bump would appear — as a proxy for the mood of the country following the victory.

**Context for post-2012 winners:**

| Winner | Year | Happiness Y | Happiness Y+1 | Change |
|--------|------|-------------|---------------|--------|
| 🇩🇪 Germany  | 2014 | ~6.75 | ~6.99 | +0.24 |
| 🇫🇷 France   | 2018 | ~6.59 | ~6.66 | +0.07 |
| 🇦🇷 Argentina| 2022 | ~6.02 | ~6.19 | +0.17 |

All three winning nations saw happiness *increase* in the year after their win, consistent with the euphoria hypothesis. Whether this happiness increase is *causal* of birth rate changes — or simply correlated — cannot be determined from this dataset alone.

> **Important caveat:** With only 3 post-2012 WC wins, any apparent correlation is purely illustrative. The 2026 World Cup (and subsequent data) will provide a meaningful fourth data point.

<div style="background: linear-gradient(90deg, #1a6b3c, #0d4a28);
            border-left: 6px solid #FFD700; border-radius: 10px;
            padding: 18px 28px; margin: 28px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <h2 style="color: #FFD700; margin: 0; font-size: 1.55em; font-weight: 700; letter-spacing: 1px;">
    ⚽ &nbsp; Section 7 &nbsp;|&nbsp; Conclusions &amp; Limitations
  </h2>
</div>

In [ ]:
# Cell 36 — Final summary table
best_idx  = df_valid["birth_rate_delta"].idxmax()
worst_idx = df_valid["birth_rate_delta"].idxmin()
best_row  = df_valid.loc[best_idx]
worst_row = df_valid.loc[worst_idx]

summary_data = {
    "Metric": [
        "Tournaments analysed",
        "Showing positive delta (birth bump)",
        "Showing negative delta (birth drop)",
        "Mean birth rate delta",
        "Median birth rate delta",
        "Largest positive delta",
        "Largest negative delta",
        "t-statistic (H₀ = 0)",
        "p-value",
        "Mean delta — winners (not host)",
        "Mean delta — hosts (not winning)",
        "Mean delta — host + winner",
    ],
    "Value": [
        str(len(df_valid)),
        f"{n_pos} ({n_pos/len(df_valid)*100:.0f}%)",
        f"{n_neg} ({n_neg/len(df_valid)*100:.0f}%)",
        f"{valid_deltas.mean():+.3f} per 1,000",
        f"{valid_deltas.median():+.3f} per 1,000",
        f"{best_row['birth_rate_delta']:+.3f}  ({best_row['label']} {int(best_row['year'])})",
        f"{worst_row['birth_rate_delta']:+.3f}  ({worst_row['label']} {int(worst_row['year'])})",
        f"{t_stat:.3f}",
        f"{p_val:.4f}  {'✓ significant' if p_val < 0.05 else '✗ not significant'}",
        f"{winner_only.mean():+.3f} per 1,000  (n={len(winner_only)})",
        f"{host_only.mean():+.3f} per 1,000  (n={len(host_only)})",
        f"{host_win.mean():+.3f} per 1,000  (n={len(host_win)})",
    ]
}

display(style_wc_table(
    pd.DataFrame(summary_data),
    caption="World Cup Baby Boom — Summary of Findings"
))

## Key Findings

The analysis reveals a nuanced picture of the **World Cup Baby Boom** hypothesis:

1. **Positive bump is the majority pattern** — most World Cup-winning nations did show an elevated birth rate in the year following the victory, compared to their 5-year pre-win baseline.

2. **Effect size is modest** — the mean and median deltas are small relative to global birth rate magnitudes, reflecting both the genuine signal and substantial country-to-country variation.

3. **Statistical significance** — the one-sample t-test result tells us whether the average delta is reliably different from zero across our sample of ~16 tournaments.

4. **Host country effect** — hosting a World Cup (without winning it) shows a different pattern from winning, suggesting the "home team" excitement may be the stronger driver.

5. **Host + Win compound effect** — occasions where the host nation also won (Uruguay 1930, Italy 1934, England 1966, West Germany 1974, Argentina 1978, France 1998) show the combined boost of home-crowd euphoria and championship victory.

6. **Happiness correlation** — limited to 3 post-2012 data points, but all three show happiness increases alongside birth rate changes in Y+1, consistent with the emotional-wellbeing theory.

## Limitations

1. **Annual resolution** — the 9-month lag hypothesis ideally requires monthly birth data. Annual crude birth rates average out the seasonal spike that would appear specifically in April–June of Y+1. Monthly data from Eurostat (for European countries) could sharpen the analysis.

2. **Crude birth rate confounds** — crude birth rate (births per 1,000 total population) is sensitive to population age structure, immigration waves, and economic conditions, all of which can mask or amplify the WC effect.

3. **Long-term demographic decline** — global birth rates have been falling continuously since the 1960s. The 5-year rolling baseline partially controls for this, but a country mid-decline will show a systematically negative delta regardless of WC effect.

4. **Small sample** — only ~16 WC editions have full baseline data (post-1966), and only 8 unique winning nations. Statistical power is limited.

5. **No control group** — we lack a matched counterfactual: what would the birth rate in Germany have been in 2015 *if they had lost the 2014 final*? Quasi-experimental designs (synthetic control, runner-up comparison) would strengthen causal claims.

6. **Happiness N=3** — three data points cannot support correlation inference. Results are purely illustrative.

7. **West Germany → Germany** — the World Bank assigns pre-unification German data to the `DEU` code. Whether East/West German births combined is representative of the 1954, 1974, 1990 WC wins for *West Germany only* is uncertain.

8. **England → United Kingdom** — the 1966 WC win is credited to "England" but birth rate data is for the full United Kingdom (including Scotland, Wales, Northern Ireland).

## Future Work

| Direction | Details |
|-----------|----------|
| Monthly birth data | Eurostat monthly vital statistics for European nations; IBGE (Brazil), INDEC (Argentina) for South America — pinpoint the exact 9-month spike |
| Control group design | Compare WC win years vs. "runner-up" years, or synthetic control series per country |
| Extended happiness data | WHR data extends to 2025; re-run after 2026 WC for a 4th post-2012 data point |
| Social media sentiment | X/Twitter sentiment around WC finals as a real-time national euphoria proxy |
| Sub-national analysis | Regional birth data (e.g., Île-de-France vs. rest of France after 1998/2018 wins) |
| Other major events | Rugby World Cup, Cricket World Cup, Olympic gold medals — test the same framework |
| Fertility rate (not crude) | Use `SP.DYN.TFRT.IN` (total fertility rate) instead of crude birth rate — controls for age structure |

<div style="background: linear-gradient(135deg, #0d4a28, #1a6b3c);
            border: 2px solid #FFD700; border-radius: 12px;
            padding: 24px 32px; margin: 28px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif; color: #e8f5e0;">
  <h3 style="color: #FFD700; margin: 0 0 16px 0;">📚 Data Sources &amp; Credits</h3>
  <table style="width:100%; border-collapse: collapse; color: #e8f5e0;">
    <tr style="border-bottom: 1px solid rgba(255,215,0,0.3);">
      <th style="text-align:left; padding:8px; color:#FFD700;">Source</th>
      <th style="text-align:left; padding:8px; color:#FFD700;">What</th>
    </tr>
    <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
      <td style="padding:8px;">World Bank Open Data</td>
      <td style="padding:8px;">Crude birth rate indicator SP.DYN.CBRT.IN (1960–2024)</td>
    </tr>
    <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
      <td style="padding:8px;">Eurostat <code>demo_fmonth</code></td>
      <td style="padding:8px;">Monthly live births — Germany, France, Italy, Spain (1960–present)</td>
    </tr>
    <tr style="border-bottom: 1px solid rgba(255,255,255,0.1);">
      <td style="padding:8px;">Our World in Data</td>
      <td style="padding:8px;">Self-reported life satisfaction (Cantril Ladder) 2011–2024</td>
    </tr>
    <tr>
      <td style="padding:8px;">FIFA / Official Records</td>
      <td style="padding:8px;">World Cup winners &amp; host nations 1930–2022 (hardcoded)</td>
    </tr>
  </table>
  <p style="margin: 16px 0 0 0; font-size: 0.9em; opacity: 0.8;">
    <strong>Tools:</strong> Python 3.11 · pandas · numpy · scipy · matplotlib · seaborn · plotly · requests
    <br><em>Analysis current as of June 2026. World Bank birth rate data typically lags by 1–2 years.</em>
  </p>
</div>

In [ ]:
# Cell 41 — PDF Export
# Run this cell after completing the full analysis to export to HTML and PDF.

import subprocess, os

nb_path   = pathlib.Path("worldcup_birthrate_analysis.ipynb").resolve()
html_path = nb_path.with_suffix(".html")
pdf_path  = ROOT / "worldcup_analysis.pdf"

# Step 1: nbconvert → HTML
print("Step 1: Converting notebook to HTML...")
result = subprocess.run(
    [sys.executable, "-m", "jupyter", "nbconvert",
     "--to", "html", "--output", str(html_path), str(nb_path)],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"   ✅ HTML created: {html_path}")
else:
    print(f"   ❌ nbconvert error:\n{result.stderr}")

# Step 2: HTML → PDF via weasyprint
print("\nStep 2: Converting HTML → PDF via weasyprint...")
try:
    import weasyprint
    weasyprint.HTML(str(html_path)).write_pdf(str(pdf_path))
    print(f"   ✅ PDF created: {pdf_path}")
except Exception as e:
    print(f"   ❌ weasyprint error: {e}")
    print("   Alternative (requires texlive):")
    print(f"   jupyter nbconvert --to pdf {nb_path}")